# Sep Agent Demo Case (ECG + MRI)

Runs the full pipeline end to end for one demo case: a real PTB-XL ECG record, a real ACDC MRI patient, and
the Sep Agent combining both contributions into one integrated decision.

**Important caveat, stated explicitly**: PTB-XL (ECG) and ACDC (MRI) share no real patients. Pairing one
record from each is a synthetic case built purely to validate the multi-agent architecture (does the shared
`AgentContribution` schema actually support cross-modality aggregation?) - it is NOT a real fused clinical
case and should not be read as one.

All three outputs (ECG contribution, MRI contribution, Sep Agent decision) are saved as JSON under
`results/sep_agent_demo/` so they can be reviewed/discussed independently of re-running this notebook.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import wfdb


def find_project_root(start=None, marker="CLAUDE.md"):
    """Walk upward from `start` (default: this notebook's cwd) until a
    directory containing `marker` is found."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (missing {marker}) from {current}")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DEMO_OUTPUT_DIR = PROJECT_ROOT / "results" / "sep_agent_demo"
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Demo output directory:", DEMO_OUTPUT_DIR)


## 1. ECG contribution

Same PTB-XL record used in `ecg_preprocessing.ipynb` (`records500/00000/00001_hr`), run through the full
signal pipeline and the ECG agent.


In [ ]:
import pandas as pd

from ecg_preprocessing import ecg_preprocessing_pipeline
from ecg_agent import generate_ecg_contribution
from agent_common import build_patient_context_line, save_contribution

PTBXL_DIR = PROJECT_ROOT / "kaggle" / "archive" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
ecg_record_path = PTBXL_DIR / "records500" / "00000" / "00001_hr"

ecg_signal, ecg_meta = wfdb.rdsamp(str(ecg_record_path))
fs = ecg_meta["fs"]
ecg_results = ecg_preprocessing_pipeline(ecg_signal[:, 0], fs)

# Patient-context line: fed into the ECG agent's own prompt via the
# existing patient_info parameter (same mechanism as before, just now
# populated from real data instead of left empty), and also reused for
# the PDF report below. Sourced directly from PTB-XL's own database.csv
# columns for this record - height happens to be missing for this
# specific record, a real example of build_patient_context_line's
# graceful-omission behavior, not a bug.
ptbxl_db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv")
ecg_row = ptbxl_db[ptbxl_db["filename_hr"].str.contains("00001_hr")].iloc[0]
ecg_patient_context = build_patient_context_line(
    age=ecg_row["age"],
    sex=ecg_row["sex"],
    height=None if pd.isna(ecg_row["height"]) else ecg_row["height"],
    weight=None if pd.isna(ecg_row["weight"]) else ecg_row["weight"],
)
print("ECG patient context:", ecg_patient_context)

ecg_contribution = generate_ecg_contribution(ecg_results, fs, record_id="00001_hr", patient_info=ecg_patient_context)
print(ecg_contribution)

ecg_output_path = save_contribution(ecg_contribution, output_dir=str(DEMO_OUTPUT_DIR), file_name="ecg_contribution")
print("Saved to:", ecg_output_path)


## 2. MRI contribution

Same ACDC patient used in `mri_preprocessing.ipynb` (`patient001`, group DCM).


In [ ]:
from mri_preprocessing import load_patient_info, preprocess_patient
from mri_agent import generate_mri_contribution
from agent_common import build_patient_context_line

ACDC_DIR = PROJECT_ROOT / "ACDC" / "database"
mri_patient_dir = ACDC_DIR / "training" / "patient001"

mri_result = preprocess_patient(str(mri_patient_dir))

# Same mechanism as the ECG cell above - fed into the MRI agent's own
# prompt via patient_info, and reused for the PDF report below. ACDC's
# Info.cfg never records age/sex (only Height/Weight), so this always
# states "age/sex not available" rather than silently omitting it - see
# build_patient_context_line's docstring for why that distinction matters.
mri_info = load_patient_info(str(mri_patient_dir / "Info.cfg"))
mri_patient_context = build_patient_context_line(height=mri_info["Height"], weight=mri_info["Weight"])
print("MRI patient context:", mri_patient_context)

mri_contribution = generate_mri_contribution(mri_result, image_mode="ed_only", patient_info=mri_patient_context)
print(mri_contribution)

mri_output_path = save_contribution(mri_contribution, output_dir=str(DEMO_OUTPUT_DIR), file_name="mri_contribution")
print("Saved to:", mri_output_path)


## 3. Sep Agent decision

Combines both contributions above into one integrated impression with per-statement traceability.


In [ ]:
from sep_agent import generate_sep_agent_decision, save_sep_agent_decision

decision = generate_sep_agent_decision(
    [ecg_contribution, mri_contribution],
    record_id="demo_case_ecg00001hr_mriPatient001",
)

print("Contributing agents:", decision.contributing_agents)
print("Flags for review (union):", decision.flags_for_review)
print()
print("SUMMARY:")
print(decision.summary)
print()
print("TRACEABILITY:")
for entry in decision.traceability:
    print(f'  "{entry.statement}" <- {entry.agent_id}.{entry.feature}')

decision_output_path = save_sep_agent_decision(
    decision, output_dir=str(DEMO_OUTPUT_DIR), file_name="sep_agent_decision"
)
print()
print("Saved to:", decision_output_path)


## Output files for review

- `results/sep_agent_demo/ecg_contribution.json` - ECG agent's structured findings
- `results/sep_agent_demo/mri_contribution.json` - MRI agent's structured findings
- `results/sep_agent_demo/sep_agent_decision.json` - Sep Agent's integrated decision + traceability

Reminder: this is a synthetic ECG+MRI pairing for architecture validation, not a real joint patient case.


## 4. PDF report

Human-readable combined report (summary, flags, traceability, per-agent findings, MRI figure) for review
with a supervisor - a presentation layer over the same data already saved as JSON above, no new analysis.


In [ ]:
from report_pdf import generate_case_report_pdf

mri_figure_path = PROJECT_ROOT / "results" / "records" / "patient001" / "mri_pipeline" / "patient001_ed_es_roi.png"

pdf_path = DEMO_OUTPUT_DIR / "case_report.pdf"
generate_case_report_pdf(
    output_path=pdf_path,
    sep_decision=decision,
    contributions=[ecg_contribution, mri_contribution],
    figure_paths=[mri_figure_path] if mri_figure_path.exists() else None,
    synthetic_pairing_note=(
        "This case pairs a PTB-XL ECG record with an ACDC MRI patient. These datasets share no real "
        "patients - this is a synthetic pairing for multi-agent architecture validation, not a real "
        "joint clinical case."
    ),
    # Rendered under each agent's own findings section only (see
    # report_pdf.py) - NOT fed into the Sep Agent's own prompt, since it
    # only combines findings the modality agents already interpreted and
    # never needs the raw demographics itself.
    patient_context_lines={"ecg": ecg_patient_context, "mri": mri_patient_context},
)
print("PDF report saved to:", pdf_path)
